# 22_model_classical_timeseries_JSC

- 목적: Naive persistence·Ridge-VAR·Kalman innovation·Isolation Forest를 같은 1초 입력 계약으로 비교해 딥러닝 전 시계열·분포 기준선을 완성한다.
- 입력: `data/processed/11_window_features.parquet`, 표준 `mean` 전처리 원신호
- 출력: `figures/22_model_classical_timeseries_JSC/`, `results/22_model_classical_timeseries_JSC/`
- 담당: JSC
- 심사 대응: 2번(4개 모델 비교), 3번(FP·FN 및 운전 상태), 4번(causal 후보), 6번(재현성)

실제 이상은 날짜가 다른 이벤트 1건이므로 AUC만으로 채택하지 않는다. `time_block fpr_segment`, 탐지 세그먼트, 합성 이상, 상태별 FP, 실시간성을 함께 본다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
import benchmark_jsc as B
import models_jsc as M
NB = "22_model_classical_timeseries_JSC"
FIG, RES = paths.nb_dirs(NB)
MODEL_DIR = paths.MODELS / NB
SEED, LOOKBACK = 0, 5
KEYS = ["naive_persistence", "ridge_var", "kalman_innovation", "iforest"]
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT), "device: CPU")

22_model_classical_timeseries_JSC figures\22_model_classical_timeseries_JSC results\22_model_classical_timeseries_JSC device: CPU


## 1. pull 반영 및 전처리 위치

13번 DC 재점검 결과를 반영한다. 기존 24~27과 공정하게 비교하는 주 결과는 파일 형식 누수를 차단한 세그먼트 평균 제거(`benchmark_v1`)를 사용한다. Ridge·Kalman의 예측 구조는 과거와 현재만 쓰지만, `mean` 전처리는 세그먼트 전체를 보므로 이 결과 자체를 완전한 실시간 성능이라고 부르지 않는다. 13번이 제안한 causal `baseline`은 이상 날짜의 DC가 남아 실제 이상 점수가 과대평가될 수 있어 Stage 4에서 별도 검증한다.

## 2. 고정 입력·평가 계약

| 항목 | 고정값 |
|---|---|
| 원신호 입력 | 10 Hz, 3채널, 1초 윈도우 `(10, 3)`, step 0.5초 |
| Ridge 문맥 | 과거 5샘플 × 3채널 → 다음 3채널 |
| Isolation Forest | 1초 진폭 피처 `amp18` |
| 학습 | 정상 데이터만 |
| 분할 | 세그먼트 GroupKFold 5 + 정상 시간순 forward block 4 |
| 임계 | 각 fold의 train 정상 score q99 |
| 커버리지 | 1초 윈도우가 있는 이상 17/21 세그먼트 |
| 합성 이상 | spike·amplitude·antiphase·current × 강도 4단계 |

Ridge의 `alpha ∈ {0.01, 0.1, 1, 10, 100}`는 각 외부 fold의 학습 정상 세그먼트 뒤 20% validation MSE로만 고르고 전체 학습 정상에 재적합한다. 이상 라벨은 선택에 쓰지 않는다.

In [2]:
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
if not PQ.exists():
    raise FileNotFoundError("11_window_features.parquet가 없습니다. 02 → 11 노트북을 먼저 실행하세요.")
segs, W = B.load_inputs()
W1 = W[W["win_s"] == 1.0]
coverage = W1.groupby("label").agg(n_windows=("seg_uid", "size"), n_segments=("seg_uid", "nunique"))
print("세그먼트 배열", len(segs), "| 1초 윈도우", len(W1))
print(coverage.rename(index={0: "정상", 1: "이상"}).to_string())
print("fold", sorted(W1["fold"].unique()), "| 정상 block", sorted(W1.loc[W1.label == 0, "block"].dropna().unique()))

세그먼트 배열 620 | 1초 윈도우 3360
       n_windows  n_segments
label                       
정상          3264         530
이상            96          17
fold [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | 정상 block [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## 3. 모델 역할

- **Naive persistence**: 직전 관측을 다음 값으로 사용한다. 모든 예측 모델의 최저 기준선이다.
- **Ridge-VAR**: 정상 파형이 선형 자기회귀만으로 충분한지 확인한다. 잔차는 학습 정상 채널별 표준편차로 기준화한다.
- **Kalman innovation**: 정상 VAR(1) 전이와 공분산으로 예측 불확실성까지 반영한 Mahalanobis innovation을 점수로 쓴다. 세그먼트마다 filter state를 초기화한다.
- **Isolation Forest `[amp18]`**: MCD의 타원형 공분산 가정과 다른 tree isolation 분포 모델이다. `contamination` 임계를 쓰지 않고 공통 train q99를 쓴다.

네 모델은 epoch가 없으므로 1 epoch smoke가 아니라 전체 9분할을 바로 실행한다.

In [3]:
cv, errors, scores = B.run(KEYS, NB, win_list=(1.0,), seed=SEED, lookback=LOOKBACK, retrain=False, segs=segs, wins=W)
metrics = B.to_metrics(cv)
details = M.collect_model_details(MODEL_DIR)
cv.to_csv(RES / "cv_scores.csv", index=False)
metrics.to_csv(RES / "metrics.csv", index=False)
errors.to_csv(RES / "error_cases.csv", index=False)
scores.to_csv(RES / "scores.csv", index=False)
details.to_csv(RES / "model_details.csv", index=False)
print("실행", len(cv), "fits | 오류 사례", len(errors), "| 점수 행", len(scores), "| 총 초", round(cv["sec"].sum(), 2))

실행 36 fits | 오류 사례 229 | 점수 행 25080 | 총 초 52.46


## 4. 전체 결과

In [4]:
summary = B.summary(cv)
summary.to_csv(RES / "summary.csv")
display(summary)
print("FN")
display(errors[errors["type"] == "FN"].groupby(["model_id", "split"])["seg_uid"].agg(list))
print("Ridge alpha 선택 횟수")
display(details[details["model"] == "ridge_var"]["selected_alpha"].value_counts(dropna=False).sort_index())

,,auc,fpr_sample,fpr_segment,n_detected,n_out_seg,delay_median_s,fpr_high_load,fpr_low_load,inj_mean,sec
model,split,,,,,,,,,,
Naive persistence,group_kfold_seg,0.8221,0.0097,0.0434,3.0,3.4,0.95,0.0216,0.0000,0.1084,0.3672
Ridge-VAR,group_kfold_seg,1.0000,0.0116,0.0280,3.4,3.4,0.90,0.0235,0.0016,0.2763,0.7544
Kalman innovation,group_kfold_seg,0.9040,0.0112,0.0317,3.0,3.4,0.95,0.0253,0.0000,0.1171,4.0438
Isolation Forest [amp],group_kfold_seg,0.9878,0.0108,0.0446,3.2,3.4,0.95,0.0204,0.0031,0.1249,0.8416
Naive persistence,time_block,0.8140,0.0275,0.1096,15.0,17.0,0.90,0.0546,0.0000,0.1143,0.2968
Ridge-VAR,time_block,1.0000,0.0060,0.0145,17.0,17.0,0.90,0.0184,0.0000,0.2857,0.6835
Kalman innovation,time_block,0.8925,0.0298,0.0879,15.0,17.0,0.90,0.0612,0.0000,0.1324,3.8052
Isolation Forest [amp],time_block,0.9851,0.0203,0.0925,16.0,17.0,0.90,0.0307,0.0105,0.1138,0.8212


FN


model_id               split          
iforest_amp_w1s        group_kfold_seg                                         [outlier_19]
                       time_block          [outlier_19, outlier_19, outlier_19, outlier_19]
kalman_innovation_w1s  group_kfold_seg                             [outlier_19, outlier_20]
                       time_block         [outlier_19, outlier_20, outlier_19, outlier_2...
naive_persistence_w1s  group_kfold_seg                             [outlier_19, outlier_20]
                       time_block         [outlier_19, outlier_20, outlier_19, outlier_2...
Name: seg_uid, dtype: object

Ridge alpha 선택 횟수


selected_alpha
0.01      5
1.00      3
100.00    1
Name: count, dtype: int64

## 5. 기존 대표 모델과 직접 비교

기존 24 DeepAnT와 26 MCD `[amp]`의 같은 1초·9분할 상세 결과를 함께 집계한다. 이 비교에서 TCN 필요성을 사전 규칙으로 판정한다.

In [5]:
ref24 = pd.read_csv(paths.RESULTS / "24_model_forecasting_JIW" / "cv_scores.csv")
ref26 = pd.read_csv(paths.RESULTS / "26_model_distribution_JIW" / "cv_scores.csv")
refs = pd.concat([
    ref24[(ref24.model == "CNN (DeepAnT)") & (ref24.win_s == 1.0)],
    ref26[(ref26.model == "MCD [amp]") & (ref26.win_s == 1.0)],
])
cols = ["model", "split", "auc", "fpr_sample", "fpr_segment", "n_detected", "n_out_seg", "inj_mean", "sec"]
comparison = pd.concat([cv[cols], refs[cols]]).groupby(["model", "split"], sort=False).mean().reset_index()
comparison.to_csv(RES / "comparison_with_references.csv", index=False)
display(comparison.round(4))

tb = comparison[comparison.split == "time_block"].set_index("model")
deep = tb.loc["CNN (DeepAnT)"]
linear = tb.loc[["Ridge-VAR", "Kalman innovation"]].copy()
linear["near_fpr"] = linear.fpr_segment <= deep.fpr_segment + 0.01
linear["near_injection"] = linear.inj_mean >= deep.inj_mean - 0.03
linear["full_detection"] = linear.n_detected >= deep.n_detected
linear["passes_all"] = linear[["near_fpr", "near_injection", "full_detection"]].all(axis=1)
need_tcn = not bool(linear["passes_all"].any())
decision = pd.DataFrame([{
    "decision": "TCN 진행" if need_tcn else "TCN 보류",
    "rule": "선형 후보가 DeepAnT 대비 fpr_segment +0.01 이내, inj_mean -0.03 이내, 탐지 수 동률을 동시에 만족하는가",
    "passing_models": ", ".join(linear.index[linear.passes_all]),
}])
decision.to_csv(RES / "tcn_decision.csv", index=False)
display(linear.round(4))
display(decision)

,model,split,auc,fpr_sample,fpr_segment,n_detected,n_out_seg,inj_mean,sec
0,Naive persistence,group_kfold_seg,0.8221,0.0097,0.0434,3.0,3.4,0.1084,0.3672
1,Ridge-VAR,group_kfold_seg,1.0000,0.0116,0.0280,3.4,3.4,0.2763,0.7544
2,Kalman innovation,group_kfold_seg,0.9040,0.0112,0.0317,3.0,3.4,0.1171,4.0438
3,Isolation Forest [amp],group_kfold_seg,0.9878,0.0108,0.0446,3.2,3.4,0.1249,0.8416
4,Naive persistence,time_block,0.8140,0.0275,0.1096,15.0,17.0,0.1143,0.2968
5,Ridge-VAR,time_block,1.0000,0.0060,0.0145,17.0,17.0,0.2857,0.6835
6,Kalman innovation,time_block,0.8925,0.0298,0.0879,15.0,17.0,0.1324,3.8052
7,Isolation Forest [amp],time_block,0.9851,0.0203,0.0925,16.0,17.0,0.1138,0.8212
8,CNN (DeepAnT),group_kfold_seg,0.9999,0.0156,0.0317,3.4,3.4,0.2502,2.0000
9,CNN (DeepAnT),time_block,0.9999,0.0137,0.0387,17.0,17.0,0.2538,1.9000


,split,auc,fpr_sample,fpr_segment,n_detected,n_out_seg,inj_mean,sec,near_fpr,near_injection,full_detection,passes_all
model,,,,,,,,,,,,
Ridge-VAR,time_block,1.0000,0.0060,0.0145,17.0,17.0,0.2857,0.6835,True,True,True,True
Kalman innovation,time_block,0.8925,0.0298,0.0879,15.0,17.0,0.1324,3.8052,False,False,False,False


,decision,rule,passing_models
0,TCN 보류,"선형 후보가 DeepAnT 대비 fpr_segment +0.01 이내, inj_me...",Ridge-VAR


## 6. 오류·강건성 시각화

In [6]:
B.plot_overview(cv, FIG)
B.plot_state_fpr(cv, FIG)
B.plot_injection(cv, FIG)
B.plot_score_margin(scores, FIG)
print(sorted(p.name for p in FIG.glob("*.png")))

Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


['fpr_by_state.png', 'injection.png', 'metrics_overview.png', 'score_margin.png']


## 7. 해석 원칙과 한계

- 이상 21세그먼트는 독립 고장 21건이 아니라 한 날짜·한 이벤트의 burst다. 1초 판정 가능 세그먼트는 17개다.
- `mean` 전처리는 기존 벤치마크 비교에는 적합하지만 세그먼트 후반을 미리 보는 오프라인 처리다. 구조가 causal인 Ridge·Kalman도 이 입력 결과만으로 실시간 모델이라고 확정하지 않는다.
- Ridge alpha 이외의 하이퍼파라미터는 이상 점수로 조정하지 않았다. TCN 진행 판정은 모델 선택 결론이지 실제 이상 AUC를 최적화하는 튜닝이 아니다.
- 합성 이상은 실제 결함 재현이 아니라 약한 이상에 대한 민감도 stress test다. 실제 이상·합성 이상·정상 FP를 함께 읽는다.
- 가중치는 git 제외 `models/22_model_classical_timeseries_JSC/`에 저장하며 노트북 실행으로 재생성한다.